# Data preprocessing

Evaluate duplicate records and apply the approved missing-value treatment.
Exploratory findings are documented in `data_exploration.ipynb`.

Run this notebook from the `notebooks/` directory with Python and pandas installed. The input is `../data/raw/student_data.csv`. Each notebook loads the input independently. The previous user environment used Python 3.14.10 and pandas 3.1.1. Execution outputs below record the validation runtime.

### Environment check

The Python interpreter, version, working directory, and pandas version
are displayed to record the execution environment.

In [1]:
import sys
import os
import pandas as pd

print(sys.executable)
print(sys.version)
print(os.getcwd())
print(pd.__version__)

/opt/homebrew/opt/python@3.15/bin/python3.15
3.15.0 (main, Oct  9 2026, 11:16:41) [Clang 17.0.0 (clang-1700.6.4.2)]
/Users/yassir/Data-Mining-Project/notebooks
3.0.6


### Data loading and initial dimensions

The CSV is loaded into the DataFrame `data`.
The loading options preserve the literal education category "None"
while treating empty fields as missing values.

The imported dataset contains 649 rows and 34 columns.
One column is an exported row index, so these are not 34 analytical features.

In [2]:
data = pd.read_csv(
    '../data/raw/student_data.csv',
    keep_default_na=False,
    na_values=['']
)

data.shape

(649, 34)

### Student variables

The exported row index is excluded to create the table of student
variables used for preprocessing.

In [3]:
student_data = data.drop(columns=['Unnamed: 0'])

student_data.shape

(649, 33)

### Missing values before preprocessing

Missing counts and percentages are recorded before applying the selected treatment.

In [4]:
missing_counts = data.isna().sum()
missing_percentages = (missing_counts / len(data) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing count': missing_counts,
    'Missing (%)': missing_percentages
})

missing_summary

,Missing count,Missing (%)
Unnamed: 0,0,0.00
school,1,0.15
sex,10,1.54
age,0,0.00
address,17,2.62
famsize,3,0.46
Pstatus,28,4.31
Medu,27,4.16
Fedu,19,2.93
Mjob,6,0.92


### Treatment of missing absences and failures

A working copy of the dataset is created for preprocessing.
Missing absences and previous failures are replaced with zero,
assuming that blanks represent no absences or failures.
This interpretation remains subject to source confirmation.

In [5]:
student_data_clean = student_data.copy()

student_data_clean['absences'] = student_data_clean['absences'].fillna(0)
student_data_clean['failures'] = student_data_clean['failures'].fillna(0)

### Validation of zero replacement

Before-and-after counts verify the replacement of missing absences and failures with zero.

In [6]:
replacement_check = pd.DataFrame({
    'Missing before': student_data[['absences', 'failures']].isna().sum(),
    'Missing after': student_data_clean[['absences', 'failures']].isna().sum(),
    'Zeros after': student_data_clean[['absences', 'failures']].eq(0).sum()
})

replacement_check

,Missing before,Missing after,Zeros after
absences,260,0,260
failures,551,0,551


### Row preservation and remaining missing values

Row counts and order are checked after filling. Remaining missing values
are displayed for later preprocessing decisions.

In [7]:
assert student_data_clean.shape == student_data.shape
assert student_data_clean.index.equals(student_data.index)
assert student_data_clean[['absences', 'failures']].isna().sum().sum() == 0

student_data_clean.isna().sum()

school         1
sex           10
age            0
address       17
famsize        3
Pstatus       28
Medu          27
Fedu          19
Mjob           6
Fjob          18
reason         7
guardian       8
traveltime    11
studytime     14
failures       0
schoolsup     30
famsup         7
paid          31
activities    31
nursery       20
higher        21
internet       5
romantic       1
famrel        24
freetime      20
goout         31
Dalc          24
Walc          27
health        17
absences       0
G1             0
G2             0
G3             0
dtype: int64

### Order of remaining treatments

Remaining missing values are sorted from most to least frequent.
Variables are treated in this order, and variables of the same type
share one treatment.

In [8]:
remaining_missing = student_data_clean.isna().sum()

remaining_missing = remaining_missing[remaining_missing > 0].sort_values(
    ascending=False
)

remaining_missing

paid          31
goout         31
activities    31
schoolsup     30
Pstatus       28
Walc          27
Medu          27
Dalc          24
famrel        24
higher        21
freetime      20
nursery       20
Fedu          19
Fjob          18
health        17
address       17
studytime     14
traveltime    11
sex           10
guardian       8
famsup         7
reason         7
Mjob           6
internet       5
famsize        3
romantic       1
school         1
dtype: int64

### Treatment of missing Yes/No responses

The eight Yes/No features contain between 1 (`romantic`) and 31
(`paid`, `activities`) missing responses. Missing responses are recorded
as a separate "Unknown" category, so no answer is guessed for a student.

`data_exploration.ipynb` (Missing-value overlap in Yes/No features) shows
that missing responses rarely occur together. Of the 140 students missing
at least one Yes/No response, 135 are missing exactly one, and the largest
pairwise overlap involves two students. "Unknown" therefore marks
individual unanswered questions rather than a group of students who left
the questionnaire incomplete.

"Unknown" adds a third level to each Yes/No feature. This must be
considered when the features are encoded.

In [9]:
yes_no_columns = [
    'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
]

for column in yes_no_columns:
    student_data_clean[column] = student_data_clean[column].fillna('Unknown')

### Validation of Yes/No replacement

Before-and-after counts verify that every missing response became
"Unknown" and that recorded "yes" and "no" answers are unchanged.

In [10]:
yes_no_check = pd.DataFrame({
    'Missing before': student_data[yes_no_columns].isna().sum(),
    'Missing after': student_data_clean[yes_no_columns].isna().sum(),
    'Unknown after': student_data_clean[yes_no_columns].eq('Unknown').sum()
})

assert student_data_clean[yes_no_columns].eq('yes').sum().equals(
    student_data[yes_no_columns].eq('yes').sum()
)
assert student_data_clean[yes_no_columns].eq('no').sum().equals(
    student_data[yes_no_columns].eq('no').sum()
)

yes_no_check

,Missing before,Missing after,Unknown after
schoolsup,30,0,30
famsup,7,0,7
paid,31,0,31
activities,31,0,31
nursery,20,0,20
higher,21,0,21
internet,5,0,5
romantic,1,0,1


### Study time by school

Missing study times are counted by school, and the median study time of
each school is calculated from recorded values. Every student with a
missing study time has a recorded school.

In [11]:
study_by_school = pd.DataFrame({
    'Missing studytime': student_data_clean.loc[
        student_data_clean['studytime'].isna(), 'school'
    ].value_counts(),
    'Median studytime': student_data_clean.groupby('school')['studytime'].median()
})

study_by_school

,Missing studytime,Median studytime
school,,
GP,8,203.5
MS,6,156.0


### Treatment of missing study time

GP students report more weekly study time than MS students, so a single
overall median would understate study time at GP and overstate it at MS.
Each missing value is replaced with the median of the student's school.
The median is used because study time is strongly right-skewed.

`transform('median')` returns, for every student, the median of their
school, aligned with the original rows. `fillna` uses that value only
where study time is missing.

In [12]:
school_study_median = student_data_clean.groupby('school')['studytime'].transform('median')

student_data_clean['studytime'] = student_data_clean['studytime'].fillna(school_study_median)

### Validation of study-time replacement

The filled values are displayed by school. The checks confirm that no
study time is missing and that recorded values are unchanged.

In [13]:
filled_study = student_data_clean.loc[
    student_data['studytime'].isna(),
    ['school', 'studytime']
]

assert student_data_clean['studytime'].isna().sum() == 0
assert student_data_clean['studytime'].where(
    student_data['studytime'].notna()
).equals(student_data['studytime'])

filled_study.value_counts()

school  studytime
GP      203.5        8
MS      156.0        6
Name: count, dtype: int64

### Travel time by school and residence

Missing travel times are counted by school and residence type, and the
median travel time of each combination is calculated from recorded values.
Every student with a missing travel time has a recorded school and
residence. `data_exploration.ipynb` (Travel time, residence, and school)
shows that rural students travel longer than urban students within both
schools.

In [14]:
travel_by_group = pd.DataFrame({
    'Missing traveltime': student_data_clean.loc[
        student_data_clean['traveltime'].isna(), ['school', 'address']
    ].value_counts(),
    'Median traveltime': student_data_clean.groupby(
        ['school', 'address']
    )['traveltime'].median()
})

travel_by_group['Missing traveltime'] = (
    travel_by_group['Missing traveltime'].fillna(0).astype(int)
)

travel_by_group

Missing traveltime  Median traveltime
school address                                       
GP     R                         1               19.0
       U                         7                9.0
MS     R                         3               25.0
       U                         0               17.0

### Treatment of missing travel time

Travel time depends on the school and on where the student lives, so a
single overall median would understate rural journeys and overstate urban
ones. Each missing value is replaced with the median of students from the
same school and residence type.

In [15]:
group_travel_median = student_data_clean.groupby(
    ['school', 'address']
)['traveltime'].transform('median')

student_data_clean['traveltime'] = student_data_clean['traveltime'].fillna(group_travel_median)

### Validation of travel-time replacement

The filled values are displayed by school and residence. The checks
confirm that no travel time is missing and that recorded values are
unchanged.

In [16]:
filled_travel = student_data_clean.loc[
    student_data['traveltime'].isna(),
    ['school', 'address', 'traveltime']
]

assert student_data_clean['traveltime'].isna().sum() == 0
assert student_data_clean['traveltime'].where(
    student_data['traveltime'].notna()
).equals(student_data['traveltime'])

filled_travel.value_counts()

school  address  traveltime
GP      U        9.0           7
MS      R        25.0          3
GP      R        19.0          1
Name: count, dtype: int64

### Rating medians

The median of each 1–5 rating feature is calculated from recorded
responses before filling.

In [17]:
rating_columns = [
    'famrel', 'freetime', 'goout',
    'Dalc', 'Walc', 'health'
]

rating_medians = student_data_clean[rating_columns].median()

rating_medians

famrel      4.0
freetime    3.0
goout       3.0
Dalc        1.0
Walc        2.0
health      4.0
dtype: float64

### Treatment of missing ratings

The six rating features contain between 17 (`health`) and 31 (`goout`)
missing responses. Each missing response is replaced with the median of
its feature. The median of an ordered 1–5 scale is always a valid
response level and is not affected by extreme responses.

`data_exploration.ipynb` (Missing-value overlap in rating features) shows
that missing ratings rarely occur together. Of the 129 students missing
at least one rating, 115 are missing exactly one, and the largest
pairwise overlap involves three students. Each feature can therefore be
filled separately.

In [18]:
for column in rating_columns:
    student_data_clean[column] = student_data_clean[column].fillna(
        rating_medians[column]
    )

### Validation of rating replacement

Before-and-after counts verify that no rating is missing, that every
filled value equals the feature median, and that recorded responses are
unchanged.

In [19]:
rating_check = pd.DataFrame({
    'Missing before': student_data[rating_columns].isna().sum(),
    'Missing after': student_data_clean[rating_columns].isna().sum(),
    'Median used': rating_medians
})

assert student_data_clean[rating_columns].isna().sum().sum() == 0
assert student_data_clean[rating_columns].where(
    student_data[rating_columns].notna()
).equals(student_data[rating_columns])

rating_check

,Missing before,Missing after,Median used
famrel,24,0,4.0
freetime,20,0,3.0
goout,31,0,3.0
Dalc,24,0,1.0
Walc,27,0,2.0
health,17,0,4.0


### Treatment of missing categorical values

Missing parental status, parental education, parental occupation,
reason for choosing the school, and residence type are recorded as a
separate unknown category rather than guessed. The observed values give
no basis for inferring, for example, a parent's occupation or education.

- `Pstatus` uses "U", following the single-letter coding of "T"
  (together) and "A" (apart).
- The other features use "Unknown". In `address`, "U" already means
  urban, so a single letter would merge unknown residences with urban
  ones.
- In `Medu` and `Fedu`, "Unknown" remains distinct from the recorded
  level "None" (no formal education).

The unknown category has no position on the education scale. This must
be considered when parental education is encoded.

In [ ]:
student_data_clean['Pstatus'] = student_data_clean['Pstatus'].fillna('U')

unknown_columns = ['Medu', 'Fedu', 'Fjob', 'address', 'reason', 'Mjob']

for column in unknown_columns:
    student_data_clean[column] = student_data_clean[column].fillna('unknown')

### Validation of categorical replacement

Before-and-after counts verify that every missing value became the
unknown category and that recorded categories are unchanged.

In [21]:
category_columns = ['Pstatus'] + unknown_columns

unknown_after = student_data_clean[unknown_columns].eq('Unknown').sum()
unknown_after['Pstatus'] = student_data_clean['Pstatus'].eq('U').sum()

category_check = pd.DataFrame({
    'Missing before': student_data[category_columns].isna().sum(),
    'Missing after': student_data_clean[category_columns].isna().sum(),
    'Unknown after': unknown_after
}).loc[category_columns]

assert student_data_clean[category_columns].isna().sum().sum() == 0
assert student_data_clean[category_columns].where(
    student_data[category_columns].notna()
).equals(student_data[category_columns])

category_check

,Missing before,Missing after,Unknown after
Pstatus,28,0,28
Medu,27,0,27
Fedu,19,0,19
Fjob,18,0,18
address,17,0,17
reason,7,0,7
Mjob,6,0,6


### Treatment of missing school

One student has no recorded school. The value is replaced with the most
frequent school (the mode). With a single missing value, this has no
meaningful effect on the distribution. `.mode()` returns a Series because
several values can tie, so `[0]` selects the first.

In [22]:
school_mode = student_data_clean['school'].mode()[0]

student_data_clean['school'] = student_data_clean['school'].fillna(school_mode)

### Validation of school replacement

School counts before and after filling are compared.

In [23]:
school_check = pd.DataFrame({
    'Before': student_data['school'].value_counts(dropna=False),
    'After': student_data_clean['school'].value_counts(dropna=False)
}).fillna(0).astype(int)

assert student_data_clean['school'].isna().sum() == 0

school_check

,Before,After
school,,
GP,422,423
MS,226,226
NaN,1,0


### Working dataset

The prepared table is available as `student_data_clean` in this notebook.
Missing failures and absences are interpreted as zero by user decision.
The source meaning of those blanks remains unconfirmed. Missing Yes/No
responses are recorded as "Unknown". Missing study times are replaced
with the median of the student's school, and missing travel times with
the median of the student's school and residence type. Missing ratings
are replaced with the median of each feature. Missing parental status is
recorded as "U"; missing parental education, parental occupation,
reason, and residence are recorded as "Unknown". The missing school is
replaced with the mode. Further treatments require an explicit decision.
Duplicate records are evaluated above.